# Notebook概要（Bronze → Silver 変換）

このNotebookは、`bronze.Sale` の生データを分析・クレンジングし、品質チェック後に `silver.Sale` テーブルとして保存する処理です。

## 処理フロー

1. **Bronzeデータの把握**  
   - `bronze.Sale` をサンプリング取得し、欠損・ユニーク値・スキーマを確認します。

2. **プロファイリング/品質確認**  
   - 欠損値集計、数値列の統計、文字列列の分布を確認し、データ品質上の注意点を洗い出します。  
   - **補足（クレンジング観点）**: 欠損の発生率だけでなく、業務上の必須項目（主キー・日付・金額）への影響を優先して確認します。  
   - **補足（クレンジング観点）**: 分布確認では、極端値・ゼロ値・異常なカテゴリ偏りを見て、後続の変換ルール（除外/補正）を決めます。

3. **Silver向けクレンジングと標準化**  
   - 主キー重複/NULL除去、数値型キャスト、金額丸め、日付型変換を行います。  
   - `Calculated_Amount` と `Amount_Discrepancy` を使って金額整合性を検証します。  
   - `DataQualityFlag`（`Valid` / 異常理由）を付与し、`ProcessedTimestamp` を追加します。  
   - **補足（クレンジング観点）**: 型統一は分析・集計の再現性を高めるために重要で、文字列数値や日付フォーマット揺れを早い段階で吸収します。  
   - **補足（クレンジング観点）**: 金額丸めは計算差異のノイズを減らしますが、業務ルール（小数桁・丸め方法）を固定し、処理間で一貫させることが重要です。

4. **保存対象/除外対象の振り分け**  
   - `DataQualityFlag == "Valid"` かつ `Sale_Key` が存在するレコードのみを保存対象にします。  
   - 異常レコードは除外データとして確認できます。  
   - **補足（クレンジング観点）**: 除外データは破棄せずに保持し、件数推移・原因内訳（例: `Invalid Price`）を監視すると品質改善サイクルを回しやすくなります。  
   - **補足（クレンジング観点）**: 保存/除外の判定基準は明文化し、しきい値変更時に影響比較できるよう運用します。

5. **Silverテーブルへ書き込み**  
   - クレンジング済みデータを Delta 形式で `Tables/silver/Sale` に `overwrite` 保存します。

6. **保存結果の確認**  
   - `demo_lakehouse.silver.Sale` を読み出して結果確認します。

## 一般的なデータクレンジングの観点（参考）

- **完全性（Completeness）**: 必須項目の欠損率、主キー欠損、参照切れを管理する。
- **妥当性（Validity）**: データ型、値域、形式（日時・コード体系）が業務ルールに合っているか確認する。
- **一意性（Uniqueness）**: 重複キーや重複イベントを検知し、重複排除ルールを固定する。
- **整合性（Consistency）**: 列間の計算関係（数量×単価=金額）や日付前後関係を検証する。
- **正確性（Accuracy）**: マスタデータや外部基準との照合で誤記・異常値を検出する。
- **追跡可能性（Auditability）**: フラグ、処理時刻、除外理由を残し、再現可能な運用にする。

## 出力データのポイント

- クレンジング済みの販売データ（型統一・丸め済み）
- 品質フラグと処理時刻を持つ監査しやすいデータ

## 本ハンズオンで実施すること

1. **セル3** のパラメータ（Lakehouse名・スキーマ名など）を自身の環境の値に変更する。
2. 先頭から **1つずつセルを実行** し、各セルの実行結果を確認する。
3. **AI関数** は書籍の手順に従って実装する（Answer用Notebookも付属しております）。
4. 後続ステップに備え、**Silverスキーマの `Sale` と `City` の結合クエリ** が実行できることを確認する。


### ▶ 実行セルの説明（環境・テーブル参照の初期設定）
**処理内容**
- Lakehouse名、Bronze/Silverスキーマ名、テーブル名を定義し、FQNと保存パスを組み立てます。

**確認ポイント**
- `LAKEHOUSE_NAME`、`BRONZE_SCHEMA`、`SILVER_SCHEMA` が自身の環境に一致していること
- 生成されたFQN/パスにスペルミスがないこと

In [1]:
## ご自身の環境の値に変更してください
LAKEHOUSE_NAME = "demo_lakehouse" # レイクハウス名
BRONZE_SCHEMA = "bronze" # Bronzeスキーマ名
SILVER_SCHEMA = "silver" # Silverスキーマ名 

SALE_TABLE = "Sale"
CITY_TABLE = "City"

BRONZE_SALE_FQN = f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{SALE_TABLE}"
SILVER_SALE_FQN = f"{LAKEHOUSE_NAME}.{SILVER_SCHEMA}.{SALE_TABLE}"
BRONZE_CITY_FQN = f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{CITY_TABLE}"
SILVER_CITY_FQN = f"{LAKEHOUSE_NAME}.{SILVER_SCHEMA}.{CITY_TABLE}"

SILVER_SALE_PATH = f"Tables/{SILVER_SCHEMA}/{SALE_TABLE}"
SILVER_CITY_PATH = f"Tables/{SILVER_SCHEMA}/{CITY_TABLE}"

print("LAKEHOUSE_NAME:", LAKEHOUSE_NAME)
print("BRONZE_SCHEMA:", BRONZE_SCHEMA)
print("SILVER_SCHEMA:", SILVER_SCHEMA)

StatementMeta(, dc0bc90f-2156-4d00-8a04-68e8435f15db, 3, Finished, Available, Finished, False)

LAKEHOUSE_NAME: demo_lakehouse
BRONZE_SCHEMA: bronze
SILVER_SCHEMA: silver


### ▶ 実行セルの説明（Sales Bronzeのプロファイリング）
**処理内容**
- Bronze `Sale` テーブルのスキーマ、欠損、基本統計、文字列分布を確認し、クレンジング方針の材料を集めます。

**確認ポイント**
- 欠損値や偏りが大きい列を特定できていること
- ID/日付/金額の候補列が検出され、後続処理に利用可能であること

In [ ]:
# Salesテーブルの中身の確認・分析
sales_bronze = spark.sql(f"SELECT * FROM {BRONZE_SALE_FQN}")

# スキーマを確認
sales_bronze.printSchema()

# 基本統計量・欠損値・異常値の簡易分析
import pyspark.sql.functions as F

# 欠損値のカウント
null_counts = sales_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in sales_bronze.columns])
display(null_counts)

# 数値型列の基本統計量確認
numeric_cols = [field.name for field in sales_bronze.schema.fields if str(field.dataType) in ['IntegerType', 'LongType', 'FloatType', 'DoubleType', 'DecimalType']]
if numeric_cols:
    display(sales_bronze.select(numeric_cols).describe())

# 文字列型列のユニーク値数や分布
string_cols = [field.name for field in sales_bronze.schema.fields if str(field.dataType) == 'StringType']
for col in string_cols:
    display(sales_bronze.groupBy(col).count().orderBy(F.desc("count")).limit(10))

# クレンジングが必要そうなサンプル値も目視確認
display(sales_bronze)

# エンリッチメントの候補列の自動検出例（例：顧客や商品、日付、売上金額などがあればAI拡張の候補）
id_cols = [c for c in sales_bronze.columns if 'id' in c.lower()]
date_cols = [c for c in sales_bronze.columns if 'date' in c.lower()]
amount_cols = [c for c in sales_bronze.columns if any(keyword in c.lower() for keyword in ['amount', 'price', 'total', 'sales'])]

print("ID候補列:", id_cols)
print("日付候補列:", date_cols)
print("金額候補列:", amount_cols)

### ▶ 実行セルの説明（Salesクレンジング）
**処理内容**
- 重複/NULL除去、型変換、端数処理、金額整合チェック、日付変換、品質フラグ付与を実施します。

**確認ポイント**
- `DataQualityFlag` の判定ロジックが想定どおりに機能していること
- 数値列・日付列の型変換結果と再計算差分（`Amount_Discrepancy`）を確認すること

In [ ]:
from pyspark.sql.functions import col, round, abs, to_date, when, current_timestamp

# 金額・数量関連の各列を数値型（decimal/float/integer）に一括クレンジング
sales_bronze = spark.sql(f"SELECT * FROM {BRONZE_SALE_FQN}")

sales_silver = (
    sales_bronze
    # 重複・NULL除去
    .dropDuplicates(["Sale_Key"])
    .filter(col("Sale_Key").isNotNull())
    
    # 数値・金額列を適切な型に変換
    .withColumn("Quantity", col("Quantity").cast("integer"))
    .withColumn("Unit_Price", col("Unit_Price").cast("double"))
    .withColumn("Total_Excluding_Tax", col("Total_Excluding_Tax").cast("double"))
    .withColumn("Tax_Amount", col("Tax_Amount").cast("double"))
    .withColumn("Profit", col("Profit").cast("double"))
    .withColumn("Total_Including_Tax", col("Total_Including_Tax").cast("double"))
    .withColumn("Total_Dry_Items", col("Total_Dry_Items").cast("double"))
    .withColumn("Total_Chiller_Items", col("Total_Chiller_Items").cast("double"))
    .withColumn("Tax_Rate", col("Tax_Rate").cast("double"))
    
    # 端数処理
    .withColumn("Unit_Price", round(col("Unit_Price"), 2))
    .withColumn("Total_Excluding_Tax", round(col("Total_Excluding_Tax"), 2))
    .withColumn("Tax_Amount", round(col("Tax_Amount"), 2))
    .withColumn("Profit", round(col("Profit"), 2))
    .withColumn("Total_Including_Tax", round(col("Total_Including_Tax"), 2))
    .withColumn("Total_Dry_Items", round(col("Total_Dry_Items"), 2))
    .withColumn("Total_Chiller_Items", round(col("Total_Chiller_Items"), 2))
    .withColumn("Tax_Rate", round(col("Tax_Rate"), 2))
    
    # 金額再計算・検証
    # Calculated_Amount: 伝票明細の数量×単価で算出した理論上の金額
    .withColumn("Calculated_Amount", col("Quantity") * col("Unit_Price"))
    # Amount_Discrepancy: 格納されている金額（Total_Excluding_Tax）と計算した金額（Calculated_Amount）の差分（絶対値）
    # -> 業務上の計算ズレ・異常値・手入力ミス等を検知する
    .withColumn("Amount_Discrepancy", abs(col("Total_Excluding_Tax") - col("Calculated_Amount")))
    
    # 日付変換（指定2列）
    .withColumn("Invoice_Date_Key", to_date(col("Invoice_Date_Key")))
    .withColumn("Delivery_Date_Key", to_date(col("Delivery_Date_Key")))
    
    # データ品質フラグ付与
    .withColumn(
        "DataQualityFlag",
        when(col("Quantity") <= 0, "Invalid Quantity")
        .when(col("Unit_Price") <= 0, "Invalid Price")
        .when(col("Amount_Discrepancy") > 0.01, "Amount Mismatch")
        .otherwise("Valid")
    )
    
    # メタデータ追加
    .withColumn("ProcessedTimestamp", current_timestamp())
)

display(sales_silver)

# スキーマを確認（データ型が変換されたことを確認）
sales_silver.printSchema()

### ▶ 実行セルの説明（Sales保存対象の作成）
**処理内容**
- Bronze列を基準に保存カラムを選定し、`DataQualityFlag = Valid` のデータだけを保存対象として抽出します。

**確認ポイント**
- 除外列（品質チェック用の一時列）が保存対象に含まれていないこと
- フィルタ後の件数が極端に減っていないこと

In [ ]:
from pyspark.sql.functions import col

# Bronzeテーブルの物理カラム名一覧取得（スキーマベースで抽出）
bronze_column_names = [field.name for field in sales_bronze.schema.fields]

# Silverデータフレームの中から、「bronzeに存在」「品質チェック用の一時列やフラグ列は除外」して保存列を決定
columns_to_exclude = set(["Calculated_Amount", "Amount_Discrepancy"])  # 品質チェック等で追加したSilver固有列
columns_to_save = [c for c in bronze_column_names if c not in columns_to_exclude]
# 必要なSilver固有列（例：品質フラグ、処理日時）は追加保存
columns_to_save += ["DataQualityFlag", "ProcessedTimestamp"]

# 保存するデータ（品質=Validかつ主キーあり）
sales_silver_to_save = (
    sales_silver
    .filter((col("DataQualityFlag") == "Valid") & col("Sale_Key").isNotNull())
    .select(*columns_to_save)
)

display(sales_silver_to_save)

### ▶ 実行セルの説明（Silverスキーマ作成）
**処理内容**
- `silver` スキーマが未作成の場合に備えて、存在チェック付きで作成します。

**確認ポイント**
- エラーなくスキーマ作成SQLが実行できること
- 既存スキーマがある場合でも安全に再実行できること

In [ ]:
# silver スキーマが存在しない場合、新規作成。
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

### ▶ 実行セルの説明（Silverスキーマの確認）
**処理内容**
- `DESCRIBE SCHEMA` で `silver` スキーマが認識されているか確認します。

**確認ポイント**
- スキーマ情報が正常に返ること
- スキーマ名のスペルミスがないこと

In [ ]:
# スキーマが認識されているか確認
display(spark.sql(f"DESCRIBE SCHEMA {SILVER_SCHEMA}"))

### ▶ 実行セルの説明（Salesデータの保存）
**処理内容**
- 保存対象に絞り込んだSalesデータをSilver層へ `overwrite` 保存します。

**確認ポイント**
- 保存先（`SILVER_SALE_PATH`）が正しいこと
- `overwrite` による既存データ上書き影響を理解していること

In [ ]:
# silverスキーマ配下にテーブルとして保存
sales_silver_to_save.write.mode("overwrite").save(SILVER_SALE_PATH)


### ▶ 実行セルの説明（Sales保存結果の確認）
**処理内容**
- Silver `Sale` テーブルをサンプリング取得し、保存結果を目視確認します。

**確認ポイント**
- テーブルが正常に作成され、件数・列構成が想定どおりであること
- 品質フラグ付きで保存した内容が反映されていること

In [ ]:
# Silver Salesテーブルのサンプリングデータを取得
sales_silver_tbl_df = spark.sql(f"SELECT * FROM {SILVER_SALE_FQN} LIMIT 1000")
display(sales_silver_tbl_df)

## AI関数の実施内容

このセルでは、AI関数を使って入力データに対する推論・変換を行い、結果列を生成します。

- 対象例: 要約、分類、特徴抽出
- 出力: 元データに対してAI処理結果の列を付与

> [!WARNING]
> AI関数を含む処理は、Sparkの**遅延評価（Lazy Evaluation）**により、`save` やテーブル書き込み時にまとめて実行されます。  
> そのため、**保存時に時間がかかる**点に注意してください。

### ハンズオンでの扱い

本ハンズオンでは、実行時間と進行を考慮し、**AI関数の実処理およびテーブル保存は実施しません**。

### 参考リンク

- [AI 関数の概要（Microsoft Learn）](https://learn.microsoft.com/ja-jp/fabric/data-science/ai-functions/overview?tabs=pandas-pyspark%2Cpyspark)

### ▶ 実行セルの説明（AI翻訳エンリッチ）
**処理内容**
- `Description` 列を日本語に翻訳し、`Description_ja` として付加したAIエンリッチ済みデータを表示します。

**確認ポイント**
- 翻訳品質に不自然な表現がないこと（AI出力の目視確認）
- 追加列 `Description_ja` が正しく生成されていること

In [ ]:
### ここにAI関数処理を追加する ###

## CityテーブルのSilver化（クレンジング方針）

このセクションでは、`bronze.City` を対象に、分析しやすく監査可能な形へ標準化して `silver.City` として保存します。
`Sale` テーブルと同様に、**要件特定 → クレンジング → 品質判定 → 保存** の順で処理します。

### 実施するクレンジング内容

1. **要件特定（列プロファイリング）**  
   - 欠損/空文字の発生状況を列単位で確認し、品質課題を把握します。  
   - 主キー候補、必須文字列列、人口列、有効期間列を自動判定し、テーブル差異に追従できるようにします。

2. **標準化（型・表記ゆれの統一）**  
   - 文字列列は `trim` を実施し、空文字を `NULL` に統一します。  
   - 主キー列は重複排除・`NULL` 除去を行い、一意性を担保します。  
   - 人口列は `long` にキャストし、数値分析可能な型へ統一します。  
   - `Valid_From` / `Valid_To` は `timestamp` へ変換し、期間比較を可能にします。

3. **品質判定（DataQualityFlag）**  
   - 必須文字列の欠損、負の人口値、不正な期間（開始 > 終了）を検知して理由を付与します。  
   - 問題がないデータは `Valid` とし、`ProcessedTimestamp` を付与して処理時点を記録します。

4. **保存対象の制御と出力**  
   - `DataQualityFlag = "Valid"` のデータのみを `silver.City` に保存します。  
   - 不正データは理由別集計で確認し、データ品質改善のために原因追跡できる状態を維持します。

### 出力データのポイント

- 文字列表記・型が統一された `City` マスタ
- 品質フラグと処理時刻を持つ監査しやすいデータ
- 保存対象と除外対象を明確に分離した運用可能なSilverデータ

### ▶ 実行セルの説明（City Bronzeの現状把握）
**処理内容**
- CityのBronzeテーブルを読み込み、欠損・空文字、主キー候補、必須列候補、人口・期間列候補を分析します。

**確認ポイント**
- 検出された候補列名（キー/人口/期間）が実データに合っていること
- 欠損や空文字の分布を見てクレンジング方針が妥当か確認すること

In [ ]:
# Cityテーブルの中身確認・分析（クレンジング要件特定）
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, trim, current_timestamp, to_timestamp

city_bronze = spark.sql(f"SELECT * FROM {BRONZE_CITY_FQN}")

# スキーマを確認
city_bronze.printSchema()

# 欠損/空文字のカウント
null_blank_counts_city = city_bronze.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in city_bronze.columns
])
display(null_blank_counts_city)

# 文字列列を把握
string_cols = [f.name for f in city_bronze.schema.fields if f.dataType.simpleString() == "string"]

# 主キー候補を自動検出（存在する場合のみ適用）
key_candidates = ["City_Key", "CityID", "City_ID", "WWI_City_ID", "id"]
city_key_col = next((c for c in city_bronze.columns if c.lower() in [k.lower() for k in key_candidates]), None)

# 必須属性候補（存在する列のみチェック対象）
required_text_candidates = ["City", "State_Province", "Country", "Continent"]
required_text_cols = [c for c in required_text_candidates if c in city_bronze.columns]

# 人口列候補
population_candidates = ["Latest_Recorded_Population", "Population"]
population_col = next((c for c in population_candidates if c in city_bronze.columns), None)

# 期間列候補
valid_from_candidates = ["Valid_From", "valid_from"]
valid_to_candidates = ["Valid_To", "valid_to"]
valid_from_col = next((c for c in valid_from_candidates if c in city_bronze.columns), None)
valid_to_col = next((c for c in valid_to_candidates if c in city_bronze.columns), None)

print("city_key_col:", city_key_col)
print("required_text_cols:", required_text_cols)
print("population_col:", population_col)
print("valid_from_col:", valid_from_col, ", valid_to_col:", valid_to_col)

display(city_bronze.limit(100))

### ▶ 実行セルの説明（Cityクレンジング）
**処理内容**
- 文字列の整形、主キー重複除去、型変換、妥当性チェックを行い、品質フラグと処理時刻を付与します。

**確認ポイント**
- `DataQualityFlag` が期待どおりに判定されていること
- 主キー欠損や重複が除去され、日時・人口列の型が適正なこと

In [ ]:
# Cityテーブルのクレンジング
city_silver = city_bronze

# 文字列列のtrim + 空文字をNULL化
for c in string_cols:
    city_silver = city_silver.withColumn(c, when(trim(col(c)) == "", None).otherwise(trim(col(c))))

# 主キー重複/NULL対応
if city_key_col:
    city_silver = city_silver.dropDuplicates([city_key_col]).filter(col(city_key_col).isNotNull())
else:
    city_silver = city_silver.dropDuplicates()

# 人口列の型統一
if population_col:
    city_silver = city_silver.withColumn(population_col, col(population_col).cast("long"))

# 期間列の型統一
if valid_from_col:
    city_silver = city_silver.withColumn(valid_from_col, to_timestamp(col(valid_from_col)))
if valid_to_col:
    city_silver = city_silver.withColumn(valid_to_col, to_timestamp(col(valid_to_col)))

# 品質フラグ
quality_expr = F.lit("Valid")
for c in required_text_cols:
    quality_expr = when(col(c).isNull(), F.lit(f"Missing {c}")).otherwise(quality_expr)

if population_col:
    quality_expr = when(col(population_col) < 0, F.lit("Invalid Population")).otherwise(quality_expr)

if valid_from_col and valid_to_col:
    quality_expr = when(col(valid_from_col) > col(valid_to_col), F.lit("Invalid Validity Period")).otherwise(quality_expr)

city_silver = (
    city_silver
    .withColumn("DataQualityFlag", quality_expr)
    .withColumn("ProcessedTimestamp", current_timestamp())
)

display(city_silver)

### ▶ 実行セルの説明（City保存対象の作成）
**処理内容**
- Bronze列を基準に保存カラムを組み立て、`DataQualityFlag = Valid` の行だけを保存対象として抽出します。

**確認ポイント**
- 保存列に必要な項目が漏れていないこと
- 無効データが除外され、件数が想定どおりになっていること

In [ ]:
# 保存列決定と保存対象の抽出
bronze_city_column_names = [field.name for field in city_bronze.schema.fields]

# Cityでは品質判定用に追加した列のみをSilver固有列として扱う
city_columns_to_save = bronze_city_column_names + ["DataQualityFlag", "ProcessedTimestamp"]
city_columns_to_save = list(dict.fromkeys(city_columns_to_save))

city_silver_to_save = (
    city_silver
    .filter(col("DataQualityFlag") == "Valid")
    .select(*city_columns_to_save)
)

display(city_silver_to_save)

### ▶ 実行セルの説明（Cityデータの保存）
**処理内容**
- `silver` スキーマを確認・作成し、クレンジング済みCityデータをSilver層に `overwrite` 保存します。

**確認ポイント**
- 保存先パス（`SILVER_CITY_PATH`）が正しいこと
- `overwrite` により既存データが置き換わる点を理解していること

In [ ]:
# silverスキーマ配下にCityテーブルとして保存
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
city_silver_to_save.write.mode("overwrite").save(SILVER_CITY_PATH)

### ▶ 実行セルの説明（City保存結果の確認）
**処理内容**
- 保存済みのSilver `City` テーブルを読み出し、サンプル表示で内容を確認します。

**確認ポイント**
- データが0件でないこと
- `DataQualityFlag` や主要属性（都市名・国名など）が正しく入っていること

In [ ]:
# 保存されたCityテーブルを確認
city_silver_tbl_df = spark.sql(f"SELECT * FROM {SILVER_CITY_FQN} LIMIT 1000")
display(city_silver_tbl_df)

このセルのSQLは、後続の **RLS（Row-Level Security）** / **CLS（Column-Level Security）** の動作確認に利用する検証クエリです。

このクエリでは `SILVER_SALE_FQN` と `SILVER_CITY_FQN` を `City_Key` で結合し、売上明細（`Sale_Key`、`Invoice_Date_Key`、`Quantity` など）と地域情報（`City`、`State_Province`、`Country`）および金額列（`Unit_Price`、`Total_Including_Tax`）を取得します。

確認ポイントは以下です。
- **RLS確認**: 実行ユーザーの権限に応じて、参照可能な行だけが返ること（地域・組織などの条件で行が制限されること）
- **CLS確認**: 権限対象外の列が非表示/マスクされること（特に金額系列の参照可否）

同一クエリをユーザー（またはロール）を切り替えて実行し、返却行数や表示列の差分を比較して動作を確認します。

### ▶ 実行セルの説明（Sales × City 結合）
**処理内容**
- Silver層の `Sale` と `City` を `City_Key` で内部結合し、分析向けの明細データを取得します。

**確認ポイント**
- 結合キー（`City_Key`）の欠損・型不一致がないこと
- 結果件数や主要列（`City`、`Country`、金額列）が期待どおりに取得できること

In [1]:
sales_city_join_df = spark.sql(f"""
SELECT
    s.Sale_Key,
    s.City_Key,
    c.City,
    c.State_Province,
    c.Sales_Territory,
    c.Country,
    c.Latest_Recorded_Population,
    s.Invoice_Date_Key,
    s.Quantity,
    s.Unit_Price,
    s.Total_Including_Tax
FROM {SILVER_SALE_FQN} AS s
INNER JOIN {SILVER_CITY_FQN} AS c
    ON s.City_Key = c.City_Key
ORDER BY s.Invoice_Date_Key DESC, s.Sale_Key DESC
""")

display(sales_city_join_df)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### ▶ 実行セルの説明（SQLマジック）
**処理内容**
- `%%sql` マジックでSQLを直接実行するためのセルです（現在は `DROP SCHEMA` がコメントアウトされています）。

**確認ポイント**
- 破壊的操作（`DROP` など）はコメントアウト状態を維持すること
- 実行する場合は対象スキーマ名と影響範囲を必ず再確認すること

In [ ]:
%%sql
-- DROP SCHEMA silver